# Обучение голоса в Kaggle — фоновый прогон

**Запускать так: Save Version → Save & Run All (Commit).** Ноутбук выполнится
целиком на серверах Kaggle: браузер, вкладку и ноутбук можно закрыть, обрыв
связи ни на что не влияет. Интерактивная сессия (Run All в редакторе), наоборот,
умирает вместе с соединением, и несохранённый результат теряется.

Перед запуском в панели справа: **Accelerator → GPU**, **Internet → On**
(интернет доступен только на аккаунте с подтверждённым телефоном), и
**+ Add Data** — датасет с вашей записью.

Лимиты: GPU-сессия до 9 часов, 30 часов GPU в неделю (сброс в субботу 00:00 UTC),
до 20 ГБ в output версии.

Ноутбук устроен так, чтобы его можно было запускать сколько угодно раз подряд:
он сам определяет, начинает с нуля или продолжает с чекпоинта прошлой версии,
и сам останавливает обучение до лимита сессии, чтобы результат успел сохраниться.


## 0. Как запущен этот ноутбук

Единственная ошибка, которая стоит дорого: запустить обучение в интерактивной
сессии и закрыть вкладку. Ячейка ниже говорит, в каком режиме вы сейчас.


In [1]:
import os

# Kaggle сообщает, как запущен ноутбук: "Interactive" (редактор) или "Batch"
# (Save & Run All). Интерактивная сессия умирает вместе с вкладкой — предупреждаем
# до того, как в неё вложены часы GPU, а не после.
RUN_TYPE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'Unknown')

if RUN_TYPE == 'Batch':
    print('фоновый прогон (Save & Run All) — вкладку можно закрывать')
elif RUN_TYPE == 'Interactive':
    print('=' * 72)
    print('ВНИМАНИЕ: это ИНТЕРАКТИВНАЯ сессия.')
    print('Она живёт, пока открыта вкладка: закроете ноутбук или порвётся связь —')
    print('сессия остановится, а всё в /kaggle/working пропадёт вместе с ней.')
    print('')
    print('Для обучения запускайте иначе: Save Version -> Save & Run All (Commit).')
    print('Тогда ноутбук считает на серверах Kaggle, и вкладка не нужна.')
    print('')
    print('Интерактивно имеет смысл только проверить установку и запись,')
    print('не доводя до обучения.')
    print('=' * 72)
else:
    print(f'тип прогона не определён (KAGGLE_KERNEL_RUN_TYPE={RUN_TYPE!r})')


фоновый прогон (Save & Run All) — вкладку можно закрывать


## Параметры


In [2]:
SPEAKER = 'anna'          # имя голоса
VC = 'rvc'                # 'rvc' (Applio) или 'sovits' (so-vits-svc-fork)
TOTAL_EPOCHS = 300        # цель суммарно, а не за один прогон
BATCH_SIZE = 8            # 4, если не хватит памяти GPU
SAVE_EVERY = 25           # как часто писать чекпоинт
TIME_BUDGET_HOURS = 7.5   # запас до лимита сессии в 9 часов
GPUS = None               # None — одна карта; 'all' — все (T4 x2)
TEXT = 'Проверка синтеза. Старинный замок на горе, а на двери замок.'


## 1. Окружение

В фоновом прогоне некому нажать «включить GPU», поэтому проверки падают сразу
и понятно.


In [3]:
import os, pathlib, shutil, subprocess, sys, time, socket, textwrap

import torch
assert torch.cuda.is_available(), 'включите Accelerator -> GPU в панели справа'
try:
    socket.create_connection(('pypi.org', 443), timeout=10).close()
except OSError:
    raise SystemExit('включите Internet -> On в панели справа')
print('GPU:', torch.cuda.get_device_name(0))

WORK = pathlib.Path('/kaggle/working')
for name in ('profiles', 'voices', 'logs'):
    (WORK / name).mkdir(exist_ok=True)
STARTED = time.time()


GPU: Tesla T4


## 2. Установка


In [4]:
!git clone --depth 1 -b claude/voice-cloning-text-synthesis-7mnlwx https://github.com/tyetladd/RustTraining /kaggle/working/RustTraining 2>/dev/null || true
%pip install -q -e '/kaggle/working/RustTraining/voice-clone-tts[stress,asr,silero]'

if VC == 'rvc':
    !git clone --depth 1 https://github.com/IAHispano/Applio /kaggle/working/Applio 2>/dev/null || true
    %pip install -q -r /kaggle/working/Applio/requirements.txt
    # Сборки torch 2.11 с PyPI собраны под CUDA 13, а образ Kaggle не кладёт
    # libcudart.so.13 туда, где его ищет torchaudio: импорт падает, и обучение
    # не стартует. Переставляем пару под CUDA 12.8 из индекса PyTorch.
    %pip install -q -U --index-url https://download.pytorch.org/whl/cu128 torch==2.11.0 torchaudio==2.11.0
    os.environ['VCTTS_APPLIO_DIR'] = '/kaggle/working/Applio'

    # git clone не тянет веса: предикторы f0, эмбеддер и предобученные
    # модели скачивает отдельная команда. Без них питч-экстракция молча
    # не пишет ни одного файла, а обучение остаётся без данных.
    !cd /kaggle/working/Applio && python3 core.py prerequisites

else:
    %pip install -q -U so-vits-svc-fork

!vctts converters


  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.7/38.7 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 66.2 MB/s eta 0:00:00
  Building editable for voice-clone-tts (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 4.3 MB/

### Проверка окружения после установки

Ворох сообщений вида «X requires Y, but you have Z» выше — это pip, ругающийся на
*другие* пакеты образа (bigframes, ydata-profiling, google-colab и прочие), которыми
мы не пользуемся. Значение имеет одно: импортируются ли `torch` и `torchaudio` в
свежем процессе — именно так их увидит обучение, которое идёт отдельными процессами.

Ячейка ниже проверяет это и говорит, что делать, если нет.


In [5]:
import json, subprocess, sys

PROBE = """
import json
out = {}
try:
    import torch
    out['torch'] = torch.__version__
    out['cuda'] = torch.cuda.is_available()
except Exception as exc:
    out['torch'] = f'ОШИБКА: {exc.__class__.__name__}: {exc}'
for name in ('torchaudio', 'torchvision'):
    try:
        out[name] = __import__(name).__version__
    except Exception as exc:
        out[name] = f'ОШИБКА: {exc.__class__.__name__}: {exc}'
print(json.dumps(out))
"""

# Важно: опрашиваем НОВЫЙ процесс. В текущем ядре torch уже импортирован, и оно
# покажет старую версию, даже если установка её заменила. Обучение тоже идёт в
# отдельных процессах — значит, видеть мы должны именно эту картину.
probe = subprocess.run([sys.executable, '-c', PROBE], capture_output=True, text=True)
fresh = json.loads(probe.stdout) if probe.stdout.strip().startswith('{') else {}
if not fresh:
    print('не удалось опросить окружение:\n', probe.stderr[-2000:])
else:
    for key, value in fresh.items():
        print(f'  {key:12}: {value}')

problems = [k for k, v in fresh.items() if str(v).startswith('ОШИБКА')]
if 'torchaudio' in problems:
    print('\n!! torchaudio не импортируется — Applio использует его для FCPE,')
    print('   обучение не стартует. Обычно это несовпадение сборки CUDA:')
    print('   %pip install -q -U --index-url https://download.pytorch.org/whl/cu128 \\')
    print('        torch==2.11.0 torchaudio==2.11.0')
elif 'torchvision' in problems:
    print('\n!! torchvision не совместим с установленным torch. Нашему пути он не нужен:')
    print('   %pip uninstall -y torchvision')
elif fresh.get('cuda') is False:
    print('\n!! CUDA не видна из свежего процесса — проверьте, что GPU включён')
else:
    print('\nокружение в порядке, можно продолжать')


  torch       : 2.11.0+cu128
  cuda        : True
  torchaudio  : 2.11.0+cu128
  torchvision : 0.25.0+cu128

окружение в порядке, можно продолжать


## 3. Запись и состояние прошлого прогона

Запись берётся из подключённого датасета. Если вы подключили output прошлой
версии (**+ Add Data → Your Work**), его `logs/`, `profiles/` и `voices/`
копируются обратно в рабочую папку — и обучение продолжится, а не начнётся заново.


In [6]:
INPUT = pathlib.Path('/kaggle/input')

audio = sorted(INPUT.rglob('*.mp3')) + sorted(INPUT.rglob('*.wav')) + sorted(INPUT.rglob('*.m4a'))
audio = [path for path in audio if 'dataset_raw' not in path.parts and path.stat().st_size > 10_000]
assert audio, 'подключите датасет с записью через + Add Data'
SOURCE = audio[0]
print('запись:', SOURCE, f'({SOURCE.stat().st_size / 1e6:.1f} МБ)')

# Состояние прошлой версии, если её подключили как входные данные.
for previous in sorted(INPUT.glob('*')):
    if not (previous / 'logs').exists():
        continue
    for name in ('logs', 'profiles', 'voices'):
        if (previous / name).exists():
            shutil.copytree(previous / name, WORK / name, dirs_exist_ok=True)
    print('подхвачено состояние из', previous)

checkpoints = list((WORK / 'logs' / SPEAKER).glob('*.pth')) if (WORK / 'logs' / SPEAKER).exists() else []
RESUME = bool(checkpoints)
print('режим:', 'продолжение' if RESUME else 'с нуля',
      f'({len(checkpoints)} чекпоинт(ов) найдено)')


запись: /kaggle/input/datasets/akm101xandrey/andrei-sample-wav/andrei-sample.wav (113.7 МБ)
режим: с нуля (0 чекпоинт(ов) найдено)


### Что показывает запись

Вердикт по длительности, формату, уровню, клиппингу и SNR — до начала обучения.
Прогон не останавливается даже при `✗`: решать вам. Но если здесь написано
«лучше переписать», то часы GPU уйдут на запись, которую всё равно придётся
переписывать.


In [7]:
cmd = f'vctts check "{SOURCE}" --purpose vc'
!{cmd}


andrei-sample.wav: запись годится

  длительность   : 19.7 мин (речи 15.2 мин)
  формат         : WAV PCM_16 48000 Гц, каналов 1, ~768 кбит/с
  уровень        : RMS -25.5 dBFS, пик -2.2 dBFS
  клиппинг       : 0.000 % сэмплов
  шум            : SNR ~36 дБ, тишины 40 %
  основной тон   : 125 Гц (male)
  дикторы Silero : aidar, eugene

  ✓ речи 15.2 мин — достаточно
  ✓ частота дискретизации 48000 Гц
  ✓ клиппинга нет
  ✓ SNR ~36 дБ


## 4. Профиль диктора

Строится один раз; при продолжении переиспользуется.


In [8]:
profile_dir = WORK / 'profiles' / SPEAKER
if (profile_dir / 'profile.json').exists():
    print('профиль уже есть, пропускаю')
else:
    cmd = f'vctts profile build "{SOURCE}" -o "{profile_dir}" --name {SPEAKER} --overwrite'
    !{cmd}


INFO voice_clone_tts.profile: loading reference audio /kaggle/input/datasets/akm101xandrey/andrei-sample-wav/andrei-sample.wav
INFO voice_clone_tts.asr: loading Whisper 'small' on cuda (float16)
INFO httpx: HTTP Request: GET https://huggingface.co/api/models/Systran/faster-whisper-small/revision/main "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: HEAD https://huggingface.co/Systran/faster-whisper-small/resolve/536b0662742c02347bc0e980a01041f333bce120/model.bin "HTTP/1.1 302 Found"
WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
INFO httpx: HTTP Request: HEAD https://huggingface.co/Systran/faster-whisper-small/resolve/536b0662742c02347bc0e980a01041f333bce120/config.json "HTTP/1.1 307 Temporary Redirect"
INFO httpx: HTTP Request: HEAD https://huggingface.co/Systran/faster-whisper-small/resolve/536b0662742c02347bc0e980a01041f333bce120/tokenizer.json "HTTP/1.1 307 Tem

## 5. Обучение

Запускается через `subprocess`, чтобы ошибку или таймаут можно было поймать:
фоновый прогон, упавший с исключением, сохраняется как **failed**, и до output
можно не добраться. Поэтому обучение ограничено бюджетом времени, а любой
неуспех не роняет ноутбук — чекпоинты остаются в output, а следующая версия
продолжит с них.


In [9]:
def train_command(resume: bool, timeout_seconds: int) -> list[str]:
    """Команда обучения; у драйверов разные полезные опции."""
    args = [
        'vctts', 'voice', 'train',
        '-p', str(WORK / 'profiles' / SPEAKER),
        '-o', str(WORK / 'voices' / SPEAKER),
        '--vc', VC,
        '--epochs', str(TOTAL_EPOCHS),
        '--resume' if resume else '--overwrite',
        '--vc-option', f'batch_size={BATCH_SIZE}',
        '--vc-option', f'timeout={timeout_seconds}',
    ]
    if VC == 'rvc':
        # Логи Applio должны лежать вне чекаута, чтобы попасть в output версии.
        if GPUS:
            # На T4 x2 Applio поднимает по процессу на карту (DDP).
            args += ['--vc-option', f'gpus={GPUS}']
        args += ['--vc-option', f'logs_dir={WORK / "logs"}',
                 '--vc-option', f'save_every_epoch={SAVE_EVERY}']
    else:
        args += ['--sample-rate', '44100']
    return args


budget = int(TIME_BUDGET_HOURS * 3600 - (time.time() - STARTED))
command = train_command(RESUME, budget)
print(' '.join(command), '\n')

result = subprocess.run(command, text=True)
TRAINED = result.returncode == 0
saved = sorted((WORK / 'logs' / SPEAKER).glob('*.pth')) if (WORK / 'logs' / SPEAKER).exists() else []
if TRAINED:
    print('\nобучение: завершено')
elif saved:
    print(f'\nобучение: прервано (код {result.returncode}), но чекпоинтов сохранено: {len(saved)}')
    print('   продолжите новой версией — она подхватит последний')
else:
    print(f'\nобучение: упало (код {result.returncode}), чекпоинтов нет')
    print('   причина — в логе выше, в строках [applio train]:')
    print('   повторный запуск с теми же настройками упрётся в то же место')


vctts voice train -p /kaggle/working/profiles/anna -o /kaggle/working/voices/anna --vc rvc --epochs 300 --overwrite --vc-option batch_size=8 --vc-option timeout=26630 --vc-option logs_dir=/kaggle/working/logs --vc-option save_every_epoch=25 



INFO voice_clone_tts.vc.dataset: dataset ready: 178 clips, 10.4 min in /kaggle/working/voices/anna/dataset/dataset_raw/anna
INFO voice_clone_tts.vc.rvc: Applio logs for 'anna' now live in /kaggle/working/logs/anna
INFO voice_clone_tts.vc.runner: running applio preprocess: /usr/bin/python3 core.py preprocess --model-name anna --dataset-path /kaggle/working/voices/anna/dataset/dataset_raw/anna --sample-rate 40000 --cpu-cores 4
INFO voice_clone_tts.vc.runner: [applio preprocess] Starting preprocess with 4 processes...
INFO voice_clone_tts.vc.runner: [applio preprocess] 100%|██████████| 178/178 [00:07<00:00, 22.76it/s]
INFO voice_clone_tts.vc.runner: [applio preprocess] Preprocess completed in 7.82 seconds on 00:10:21 seconds of audio.
INFO voice_clone_tts.vc.runner: [applio preprocess] Model anna preprocessed successfully.
INFO voice_clone_tts.vc.rvc: preprocess: 268 фрагментов в /kaggle/working/Applio/logs/anna/sliced_audios
INFO voice_clone_tts.vc.runner: running applio extract: /usr/bi

dataset     : /kaggle/working/voices/anna/dataset/dataset_raw/anna
clips       : 178 (10.4 min)
sample rate : 40000 Hz
median F0   : 129 Hz

voice model : anna
converter   : rvc
directory   : /kaggle/working/voices/anna
checkpoint  : /kaggle/working/voices/anna/G_10800.pth
speaker     : anna @ 40000 Hz
median F0   : 129 Hz
from profile: /kaggle/input/datasets/akm101xandrey/andrei-sample-wav/andrei-sample.wav
  epochs    : 300
  clips     : 178
  dataset_minutes: 10.37
  f0_method : rmvpe
  batch_size: 8

Use it with:  vctts speak -b silero --voice-model /kaggle/working/voices/anna -t "…"

обучение: завершено


## 6. Проверка: Silero TTS + ваш голос

Работает только если модель уже собрана. Если обучение прервалось по времени,
ячейка просто сообщит об этом.


In [10]:
model_dir = WORK / 'voices' / SPEAKER
sample = WORK / f'{SPEAKER}-sample.wav'

if (model_dir / 'voice_model.json').exists():
    speak = [
        'vctts', 'speak', '-b', 'silero', '-l', 'ru',
        '--voice-model', str(model_dir), '--transpose', 'auto',
        '-t', TEXT, '-o', str(sample),
    ]
    check = subprocess.run(speak, text=True)
    print('синтез:', 'ок' if check.returncode == 0 else 'не удался')
else:
    print('модели ещё нет — обучение не дошло до конца, продолжите новой версией')

if sample.exists():
    from IPython.display import Audio, display
    display(Audio(str(sample)))


INFO voice_clone_tts.text.stress: loading silero-stress accentor (lang=ru)
INFO voice_clone_tts.pipeline: prepared 1 chunk(s) in ru
INFO voice_clone_tts.pipeline: synthesizing chunk 1/1 (70 chars)
INFO voice_clone_tts.backends.silero: loading silero_tts v5_5_ru (ru)


Downloading: "https://github.com/snakers4/silero-models/zipball/master" to /root/.cache/torch/hub/master.zip


100%|██████████| 139M/139M [00:06<00:00, 23.5MB/s]
<torch_package_1>.multi_acc_v3_package.py:286: SyntaxWarning: invalid escape sequence '\^'
  text = re.sub(r'[^{}]'.format(self.symbols[3:] + '\^'), '', text)
INFO voice_clone_tts.pipeline: auto transpose: 190 Hz -> 129 Hz = -7 semitones
INFO voice_clone_tts.vc.runner: running applio infer: /usr/bin/python3 core.py infer --input-path /tmp/vctts-rvc-9ly95myd/in.wav --output-path /tmp/vctts-rvc-9ly95myd/out.wav --pth-path /kaggle/working/voices/anna/G_10800.pth --pitch -7 --f0-method rmvpe --index-rate 0.3 --protect 0.33 --volume-envelope 1.0 --index-path /kaggle/working/voices/anna/anna.index
INFO voice_clone_tts.vc.runner: [applio infer] Traceback (most recent call last):
INFO voice_clone_tts.vc.runner: [applio infer]   File "/kaggle/working/Applio/core.py", line 1336, in <module>
INFO voice_clone_tts.vc.runner: [applio infer]     main()
INFO voice_clone_tts.vc.runner: [applio infer]   File "/kaggle/working/Applio/core.py", line 1332, 

синтез: не удался


## 7. Итог и что забрать

Всё, что осталось в `/kaggle/working`, попадает в output сохранённой версии:
оттуда можно скачать архив модели, а можно подключить этот output к следующему
запуску — тогда ноутбук сам продолжит обучение.


In [11]:
# Чекаут Applio и клон репозитория в output не нужны — они занимают гигабайты.
for junk in ('Applio', 'RustTraining'):
    shutil.rmtree(WORK / junk, ignore_errors=True)

if (model_dir / 'voice_model.json').exists():
    archive = shutil.make_archive(str(WORK / f'{SPEAKER}-voice'), 'zip', model_dir)
    print('архив модели:', archive)

elapsed = (time.time() - STARTED) / 3600
print(textwrap.dedent(f'''
    итог прогона
      голос      : {SPEAKER} ({VC})
      режим      : {'продолжение' if RESUME else 'с нуля'}
      обучение   : {'завершено' if TRAINED else 'прервано, нужен ещё прогон'}
      время      : {elapsed:.1f} ч
      дальше     : {'скачайте архив из output' if TRAINED else 'запустите Save & Run All ещё раз, подключив этот output через + Add Data -> Your Work'}
'''))


архив модели: /kaggle/working/anna-voice.zip

итог прогона
  голос      : anna (rvc)
  режим      : с нуля
  обучение   : завершено
  время      : 3.6 ч
  дальше     : скачайте архив из output



---

### Если хочется следить за ходом

Открытая вкладка для фонового прогона не нужна: состояние видно в **Notebook →
версии** (Running / Complete / Failed), а полный лог — по клику на версию.
Kaggle присылает уведомление о завершении. Квота расходуется только фактическим
временем прогона.

### Если прогон упал на установке

Самое частое — Applio ставит свои версии torch и transformers поверх образа.
В фоновом прогоне это обычно проходит; если нет, попробуйте `VC = 'sovits'`:
so-vits-svc-fork ставится одной командой и не конфликтует с образом.
